# Indego Bikes: Do electric bikes tempt riders to go faster?

<center><b>Abstract</b></center>

Bike-share programs are increasingly popular worldwide. Following successful programs in Amsterdam, Paris, and Copenhagen, the City of Philadelphia launched Indego Bikes in 2015. Indego maintains a fleet of over 2,000 bicycles, more than half of which are electric bicycles. Urban cycling can be dangerous. To investigate whether riders travel faster on electric bicycles, we compared the duration of trips made using standard and electric bicycles between a pair of stations two miles apart in downtown Philadelphia. Following a pre-specified plan, we used 161 trips from the third quarter of 2023 (113 electric, 48 standard), excluding trips longer than twice Google's 13-minute estimate. Electric-bike trips averaged 0.75 minutes shorter (12.23 versus 12.98). A one-sided permutation test that shuffles bike-type labels gives p ≈ 0.068, so we cannot reject the null hypothesis at the 5% significance level; a 95% bootstrap confidence interval for the difference (−0.03 to 1.54 minutes) likewise includes zero. The result is suggestive but not significant. Even if real, the effect size is small (about 45 seconds on a two-mile ride) and likely of limited practical importance; these results do not support the conclusion that riders of electric bicycles are at greater risk solely due to speed.


![Indego riders](Indego_riders.jpg)
You are probably all familiar with [Indego Bikes](https://www.rideindego.com/), the Philly bike-share program. What you may not know is that Indego makes ridership data available on their [trip data](https://www.rideindego.com/about/data/) website. 

In this sample project, we will:
* Explore trip data for the third quarter of 2023
* Make data visualizations
* Formulate a hypothesis
* Test the hypothesis using simulation

In [ ]:
import numpy as np
from datascience import *
import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')
%matplotlib inline

## Load the data

In [ ]:
# Note: this is a large data set, hence the low_memory flag.
indego = Table.read_table('indego-trips-2023-q3-2.csv', low_memory=False)

## Exploratory Data Analysis

In [ ]:
indego.show(3)

In [ ]:
indego.num_rows

**That is a lot of bike trips!**

### Trip Duration
The data set includes the trip durations, which are measured in minutes according to the [website](https://www.rideindego.com/about/data/). Let's see how this variable is distributed. 

In [ ]:
min_duration = min(indego.column('duration'))
max_duration = max(indego.column('duration'))
print(f" The longest trip was {max_duration} minutes.")
print(f" The shortest trip was {min_duration} minutes.")

The Indego site says trips below 1 minute are removed. Probably, the rider changed their mind and put the bike back. **But honestly, can you go anywhere in one minute?  Let's see.**

There is a field "trip_route_category" that is either "Round Trip" or "One Way." If a 1-minute trip is round trip, clearly the bike never left the station. Are there any one-way trips?

In [ ]:
quick_trips = indego.where('duration', 1).where('trip_route_category', 'One Way')
quick_trips.show(5)

**Wow!** Yes, there are some really fast trips! How is that possible? Let's look at the start and end stations for the first of these trips in the table above:

start_station is 3061

end_station is 3161

We need to know the station names. Fortunately, Indego provides this in a separate CSV file.

### Load Station Name Data

In [ ]:
station_names = Table.read_table('indego-stations-2023-10-01.csv', low_memory=False)
station_names.show(3)

In [ ]:
# We just need the first two columns
station_names = station_names.select('Station_ID', 'Station_Name')
print('Start Station')
print(station_names.where('Station_ID', 3061))
print()
print('End Station')
print(station_names.where('Station_ID', 3161))

![Quick bike trip](quick_trip.jpg)

If you put these addresses into Google Maps, it says this is a two-minute ride. I can imagine a Penn or Drexel student getting off the train at 30th Street Station and making this ride in double-quick time, as it is only a third of a mile. Alternatively, the clocks at the two stations might not be perfectly in sync. Or it could be a minute and 29 seconds rounded down. Anyway, it appears feasible.

Now let's look at the overall distribution of ride durations.

In [ ]:
indego.hist('duration', bins=np.arange(0, 80, 5))

We see most trips are short, the peak being 5-10 minutes. Very few are over an hour.

### Trips by Time of Day

Let's look at the distribution of rides by time of day. This requires parsing the hour from strings such as the following.

In [ ]:
st = indego.column("start_time")
example = st[100]
example

To extract the hour, we can first split the date and time on the space.

In [ ]:
example.split()

Then we keep the second term and split on the colon.
`example.split()[1].split(":")`

In [ ]:
example.split()[1].split(":")

Hour is the first element after this second split. Finally, we convert the hour from a string to an integer so we can plot the distribution.

In [ ]:
hour = example.split()[1].split(":")[0]
hour = int(hour)
hour

Put this in a function.

In [ ]:
def extract_hour(date_time_string):
    '''
    This function expects a datetime string in the form
    '7/1/2023 1:29' and returns just the hour.
    '''
    hour = date_time_string.split()[1].split(":")[0]
    return int(hour)

In [ ]:
# Test our function
test_datetime = st[5000]
print(test_datetime)
print(extract_hour(test_datetime))

Now use our function to add a "start_hour" column to our data table.

In [ ]:
indego = indego.with_column('start_hour', indego.apply(extract_hour, 'start_time'))
indego.show(3)

In [ ]:
# Use 24 bins for the 24 hours in a day.
indego.hist('start_hour', bins=24)

So the least likely time to start an Indego ride is 3 AM (Hmmm, I wonder why :-), and peak ridership is in the late afternoon and evening with the maximum at about 6 PM (18:00 hours).

### Rides by Type of Bicycle
Indego offers two types of bikes: standard and electric. Let's see which constitutes the majority of the rides.

In [ ]:
indego_grp = indego.group('bike_type')
indego_grp

Wow! The ebikes, which Indego introduced only a few years earlier, account for slightly more than half of the rides.

This raises the question: are more riders choosing ebikes because they prefer them, or simply because there are more ebikes to ride? A first step is to find what fraction of the bike fleet is ebikes and how many rides each bike gets. We can count the bikes using the bike ID in concert with the bike type.

In [ ]:
ebike = indego.where('bike_type', 'electric')
ebike_id = np.unique(ebike.column('bike_id'))
num_ebikes = len(ebike_id)

standard = indego.where('bike_type', 'standard')
standard_id = np.unique(standard.column('bike_id'))
num_standard = len(standard_id)

print("The number of ebikes is:", num_ebikes)
print("The number of standard bikes is:", num_standard)

So there are more ebikes in the fleet. Compare ratios.

In [ ]:
rides_per_ebike = indego_grp.column('count').item(0) / num_ebikes
rides_per_ebike

In [ ]:
rides_per_standard = indego_grp.column('count').item(1) / num_standard
rides_per_standard

So even though ebikes make up more of the fleet, each standard bike was ridden a bit more often this quarter: about 177 rides per standard bike versus 155 per ebike. This ratio measures **utilization**, not preference. Many things besides what riders want affect how often a bike gets used:

* **Availability:** whether that type of bike is docked at the station when a rider arrives.
* **Charge level:** riders may pass over an ebike with a low battery, and ebikes that need charging may be out of service.
* **Price:** ebikes cost extra to ride. Here is the current [pricing information](https://www.rideindego.com/buy-a-pass/#/).

To learn which type riders actually prefer, we would need to know which bikes were available, and how charged they were, when each rider made a choice.

## Hypothesis
While thinking about ebikes and standard bikes, an interesting hypothesis occurred to me. I wondered whether riders rode faster on ebikes than standard bikes. If so, this could make ebikes more dangerous, particularly in an urban environment. 

I did some background research. Here are some references on this topic:

```
Langford, B. C., Chen, J., & Cherry, C. R. (2015). Risky riding: Naturalistic methods comparing safety behavior from conventional bicycle riders and electric bike riders. Accident Analysis & Prevention, 82, 220-226.

Gogola, M. (2018, April). Are the e-bikes more dangerous than traditional bicycles?. In 2018 XI International Science-Technical Conference Automotive Safety (pp. 1-4). IEEE.

Siman-Tov, M. (2018). A look at electric bike casualties: do they differ from the mechanical bicycle? J Transp Heal 11 (October): 176–182.
```




**Hypothesis: Riders bike at higher speeds on ebikes.**

**Null Hypothesis: Any difference in mean ride speed by type of bike can be explained by natural variability in ride times.**

To test this hypothesis, we can compare mean trip duration by bicycle type where all of the trips start and end at the same two bike stations. Presumably, the time taken to check out and return the bikes would be roughly the same, so any difference would be attributable to ride speed. Clearly, we need stations reasonably far apart so the ride isn't too short, and a pair of stations with a lot of trips to furnish an adequate sample size.

### Analysis Plan
We wrote down this plan before looking at any trip durations between our two stations. Settling these choices in advance keeps us from picking, even unintentionally, the version of the analysis that gives the result we hoped for.

**Station pair.** We compare trips in both directions between 15th & Spruce (station 3010) and Point Breeze & Tasker (station 3053). *Disclosure:* this pair was not chosen blind. In an earlier exploratory pass, we found it by taking the busiest starting station and its most popular destination among rides of 10–20 minutes, so that search did look at durations. A fully pre-specified analysis would choose the pair with a rule that never looks at durations, such as the busiest pair of stations at least a mile apart.

**Exclusion rule.** Google Maps says this route is 2.0 miles and takes about 13 minutes by bike (see the map below). A trip much longer than that probably included a stop along the way, so we drop any trip longer than 26 minutes, twice Google's estimate. The rule is the same for both bike types, and we keep every shorter trip.

**Test.** Our statistic is the mean trip duration on standard bikes minus the mean trip duration on electric bikes. Our hypothesis says this difference is positive, so the test is one-sided. We use a permutation test that shuffles the bike-type labels, and the p-value is the fraction of shuffled differences at least as large as the observed one. We will reject the null hypothesis if p < 0.05 (significance level α = 0.05). We will also report a 95% bootstrap confidence interval for the difference.

![bike_route from Google Maps](bike_route.jpg)

### Selecting the Trips
Following the plan, we take every trip between the two stations, in both directions.

In [ ]:
station_names.where('Station_ID', are.contained_in(make_array(3010, 3053)))

In [ ]:
common_trip = indego.where('start_station', 3010).where('end_station', 3053)
reverse_trip = indego.where('start_station', 3053).where('end_station', 3010)
trip = common_trip.append(reverse_trip)
trip.num_rows

Next we apply the exclusion rule from the plan: drop trips longer than twice Google's 13-minute estimate.

In [ ]:
cutoff = 2 * 13
num_dropped = trip.where('duration', are.above(cutoff)).num_rows
trip = trip.where('duration', are.below_or_equal_to(cutoff))
print(f"Dropped {num_dropped} trips longer than {cutoff} minutes; {trip.num_rows} trips remain.")

### Check for Outliers
Make a box plot. (You can learn more about box plots [here](https://statisticsbyjim.com/graphs/box-plot/).)

In [ ]:
trip.select('duration').boxplot()

The box plot shows the trips that remain after the exclusion rule. A few rides are still well above the median, but our plan said to keep every trip of 26 minutes or less, so we keep them rather than deciding after the fact which points look too slow.

Notice that the median ride duration is 12 minutes, about a minute faster than Google predicts, but as any frequent bike rider will tell you, Google Maps tends to be conservative.

### Compare standard and ebike histograms

In [ ]:
type_count = trip.group('bike_type')
type_count

In [ ]:
num_standard = trip.where('bike_type', 'standard').num_rows
num_ebike = trip.where('bike_type', 'electric').num_rows

In [ ]:
print(f"Our total sample size is {num_standard + num_ebike} trips.")

In [ ]:
trip.where('bike_type', 'standard').hist('duration')
plt.title("Standard bikes");

In [ ]:
trip.where('bike_type', 'electric').hist('duration')
plt.title("Electric Bikes");

It certainly appears the ebike riders are a bit quicker, but is the difference statistically significant?

As planned, our hypothesis is directional (riders are *faster* on ebikes), so our test statistic is the mean trip duration on standard bikes minus the mean trip duration on electric bikes. A positive value means trips on electric bikes were shorter.

In [ ]:
standard_mean_duration = np.mean(trip.where('bike_type', 'standard').column('duration'))
ebike_mean_duration = np.mean(trip.where('bike_type', 'electric').column('duration'))
print(f"The average trip on a standard bike takes {np.round(standard_mean_duration, 2)} minutes.")
print(f"The average trip on an electric bike takes {np.round(ebike_mean_duration, 2)} minutes.")
print()
print(f"The difference in means is {np.round(standard_mean_duration - ebike_mean_duration, 2)} minutes")

In [ ]:
def standard_minus_electric(tbl):
    standard_mean = np.mean(tbl.where('bike_type', 'standard').column('duration'))
    electric_mean = np.mean(tbl.where('bike_type', 'electric').column('duration'))
    return standard_mean - electric_mean

observed_difference = standard_minus_electric(trip)
observed_difference

So trips on electric bikes were shorter, but by less than a minute on average. Could this be due to chance? Run a simulation!

### Simulation Under the Null Hypothesis
If the null hypothesis is true, the type of bike has nothing to do with trip duration, so the "standard" and "electric" labels are interchangeable. We can simulate this by randomly shuffling the labels among the trips and recalculating the difference in means. This is the permutation test from [Inferential Thinking, Section 12.1](https://inferentialthinking.com/chapters/12/1/AB_Testing.html).

In [ ]:
def one_shuffled_difference():
    shuffled_labels = trip.sample(with_replacement=False).column('bike_type')
    shuffled_table = trip.select('duration').with_column('bike_type', shuffled_labels)
    return standard_minus_electric(shuffled_table)

In [ ]:
# Test function
one_shuffled_difference()

A single shuffle gives one simulated difference. We need to repeat the shuffle many times to see how the difference is distributed under the null hypothesis.

In [ ]:
# This takes about 15 seconds.
num_repetitions = 20000
differences = make_array()
for i in np.arange(num_repetitions):
    differences = np.append(differences, one_shuffled_difference())

In [ ]:
results = Table().with_column('Simulated Difference', differences)

In [ ]:
results.hist('Simulated Difference')
plt.scatter(observed_difference, 0, color='red', s=200);
plt.title('Standard minus Electric Mean Duration');

In [ ]:
# One-sided p-value: the fraction of simulated differences at least as large as the observed one
p_value = np.count_nonzero(differences >= observed_difference) / num_repetitions
p_value

### Bootstrap Confidence Interval
The p-value tells us whether the observed difference could be due to chance. To show how uncertain our estimate of the difference is, we also build a 95% confidence interval for the difference in mean trip duration (standard − electric). We bootstrap: resample each group of trips with replacement, recalculate the difference in means, and repeat many times ([Inferential Thinking, Section 13.2](https://inferentialthinking.com/chapters/13/2/Bootstrap.html)).

In [ ]:
standard_trips = trip.where('bike_type', 'standard')
electric_trips = trip.where('bike_type', 'electric')

def one_bootstrap_difference():
    standard_resample = standard_trips.sample()
    electric_resample = electric_trips.sample()
    return np.mean(standard_resample.column('duration')) - np.mean(electric_resample.column('duration'))

In [ ]:
boot_differences = make_array()
for i in np.arange(5000):
    boot_differences = np.append(boot_differences, one_bootstrap_difference())

In [ ]:
left_end = percentile(2.5, boot_differences)
right_end = percentile(97.5, boot_differences)
print(f"95% confidence interval for standard − electric: ({np.round(left_end, 2)}, {np.round(right_end, 2)}) minutes")

In [ ]:
Table().with_column('Bootstrap Difference', boot_differences).hist('Bootstrap Difference')
plt.plot([left_end, right_end], [0, 0], color='yellow', lw=8);
plt.title('Bootstrap Differences with 95% Interval');

The interval runs from about −0.03 to 1.54 minutes. Because it contains 0, it agrees with our hypothesis test: at the 5% level we cannot rule out that there is no difference. Most of the interval is above 0, though, which is consistent with trips on electric bikes being somewhat shorter, by anywhere from essentially nothing to about a minute and a half.

## Conclusions

The distributions hinted that riders might be slightly quicker on electric bikes: electric trips were about 0.75 minutes (roughly 45 seconds) shorter on average. However, when we shuffled the bike-type labels, about 7% of the simulated differences were at least as large as the one we observed (p ≈ 0.068). This is above the significance level of α = 0.05 that we set in our plan, so we cannot reject the null hypothesis. The data are consistent with riders being slightly faster on electric bikes, at least for a two-mile ride in the city, but the evidence is not strong enough to rule out chance. The 95% bootstrap confidence interval for the difference, about −0.03 to 1.54 minutes, tells the same story: it includes 0. In short, the result is suggestive, but not statistically significant.

Would more data settle the question? More trips would make our estimate more precise: the bootstrap confidence interval would get narrower. But a narrower interval could still include 0, so more data would not guarantee a statistically significant result. We could get more trips by analyzing more station pairs and by downloading more quarters of data from the Indego site. (An exercise left to the reader ;-)

The **effect size**, however, appears to be small. Does anyone care if we shave 45 seconds off of a two-mile commuter ride? Unlikely. Even results that are "statistically significant" do not always have practical importance. It does not appear the ebike riders are taking greater risks because they are not going that much faster.

## Future Work
Any good scientific study ends with a reflection on possibilities for further study. There are many ideas we could pursue:

* How does ridership change with the weather? We could download Philly weather data and compare ridership on warm, sunny days versus cold, rainy ones.
* How does ridership change with the academic calendar? Do students dominate the ridership near universities? 
* Which stations still show activity in the wee hours of the morning? Where are people riding at 3 AM?
* How has the balance between standard and ebikes changed over the last few years?
* Are there parts of the city that are under-served and could use more stations?

**Just a few of many ideas for follow-up research!**
